# 01 - Análisis del dataset de fraude Yape/BCP

**Objetivo:** explorar el dataset sintético, obtener estadísticas descriptivas, analizar la distribución del fraude y estudiar variables relevantes.

> El dataset de este proyecto es **sintético** y se utiliza con fines académicos.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "datos" / "dataset_fraude_yape.csv"
df = pd.read_csv(DATA_PATH)

print("Filas y columnas:", df.shape)
df.head()


## Información general


In [ ]:
df.info()
print("\nValores nulos:")
display(df.isnull().sum().to_frame("nulos"))
print("\nDuplicados:", df.duplicated().sum())


## Estadísticas descriptivas


In [ ]:
display(df.describe(include="all").T)
print("Promedio del monto: S/", round(df["monto"].mean(), 2))
print("Mediana del monto: S/", round(df["monto"].median(), 2))
print("Monto máximo: S/", round(df["monto"].max(), 2))


## Normal vs fraude


In [ ]:
conteo = df["fraude"].value_counts().sort_index()
porcentaje = df["fraude"].value_counts(normalize=True).sort_index() * 100
resumen = pd.DataFrame({"cantidad": conteo, "porcentaje": porcentaje.round(2)},
                       index=["Normal", "Fraude"])
display(resumen)

plt.figure(figsize=(7,5))
sns.countplot(data=df, x="fraude")
plt.title("Operaciones normales vs. fraude")
plt.xlabel("Fraude (0=Normal, 1=Fraude)")
plt.ylabel("Cantidad")
plt.show()


## Distribución de montos


In [ ]:
plt.figure(figsize=(9,5))
sns.histplot(data=df, x="monto", bins=50, kde=True)
plt.title("Distribución de montos")
plt.xlabel("Monto (S/)")
plt.ylabel("Frecuencia")
plt.show()


## Análisis por producto


In [ ]:
producto = pd.crosstab(df["producto"], df["fraude"], normalize="index") * 100
producto.columns = ["Normal (%)", "Fraude (%)"]
display(producto.round(2))

plt.figure(figsize=(9,5))
sns.countplot(data=df, x="producto", hue="fraude")
plt.title("Fraude por producto")
plt.xlabel("Producto")
plt.ylabel("Cantidad")
plt.xticks(rotation=20)
plt.show()


## Variables relacionadas con riesgo


In [ ]:
variables = ["destinatario_nuevo", "hora_inusual", "llamada_reciente",
             "cambio_dispositivo", "usuario_nuevo", "ubicacion_inusual"]
for col in variables:
    print("\n", col)
    display((pd.crosstab(df[col], df["fraude"], normalize="index") * 100).round(2))


## Correlaciones


In [ ]:
numericas = df.select_dtypes(include=np.number).drop(
    columns=["id_transaccion", "puntaje_riesgo", "fraude"], errors="ignore")
plt.figure(figsize=(12,9))
sns.heatmap(numericas.corr(), cmap="coolwarm", center=0)
plt.title("Correlación de variables numéricas")
plt.show()


## SciPy: prueba t de Welch


In [ ]:
from scipy import stats

normal = df.loc[df["fraude"] == 0, "monto"]
fraude = df.loc[df["fraude"] == 1, "monto"]
resultado = stats.ttest_ind(normal, fraude, equal_var=False)

print("Estadístico t:", round(resultado.statistic, 4))
print("p-value:", resultado.pvalue)
print("Resultado:", "diferencia significativa" if resultado.pvalue < 0.05
      else "sin diferencia significativa")


## Conclusión

El análisis permite conocer la distribución de los datos, el desbalance entre operaciones normales y fraudulentas, el comportamiento por producto y variables asociadas al riesgo. Estos resultados sirven como base para el entrenamiento.
